# Phase 6 fix — seeded from-scratch refit

The `--load-dir` warm start cannot work: splatfacto resizes its parameters when
loading a checkpoint, which disconnects them from the already-built optimizers,
so training runs but nothing learns. This notebook refits **from scratch** on
the ROSE-edited frames, seeded with the garden model's own point cloud
(sharp init, correctly wired optimizers). Run all four cells top to bottom;
cell 3 trains ~25 min (progress bar goes 0 -> 100%).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/light-footprint-removal'

In [ ]:
# Install. If Colab asks to restart: restart, re-run cell 1, skip this cell.
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
!pip -q install nerfstudio
!pip -q install "diffusers>=0.31" accelerate imageio imageio-ffmpeg
!pip -q install --force-reinstall "numpy==2.0.2"

In [ ]:
# Refit from scratch, seeded with the garden model's points
import glob, json, os, shutil
import numpy as np, torch

P6 = f'{DRIVE}/checkpoints/phase6_realdemo'

# restore anything missing (no-ops if the session is still alive)
if not glob.glob('/content/out_garden/**/*.ckpt', recursive=True):
    shutil.copytree(f'{P6}/out_garden', '/content/out_garden', dirs_exist_ok=True)
if not os.path.isdir('/content/data/edited_garden/rgb'):
    os.makedirs('/content/data/edited_garden', exist_ok=True)
    shutil.copytree(f'{P6}/edited_garden', '/content/data/edited_garden/rgb', dirs_exist_ok=True)
    shutil.copy(f'{P6}/orbit_transforms.json', '/content/data/edited_garden/transforms.json')
    meta = json.load(open('/content/data/edited_garden/transforms.json'))
    fr = meta['frames']; test_idx = set(range(0, 81, 8))
    for name, part in {'train': [f for i, f in enumerate(fr) if i not in test_idx],
                       'test':  [f for i, f in enumerate(fr) if i in test_idx],
                       'val':   [f for i, f in enumerate(fr) if i in test_idx]}.items():
        json.dump({'camera_angle_x': meta['camera_angle_x'], 'frames': part},
                  open(f'/content/data/edited_garden/transforms_{name}.json', 'w'))

# garden checkpoint -> seed point cloud (positions + colors)
ck = sorted(glob.glob('/content/out_garden/**/step-000029999.ckpt', recursive=True))[0]
sd = torch.load(ck, map_location='cpu')['pipeline']
means = sd['_model.gauss_params.means'].float().numpy()
rgb = np.clip(0.28209479177 * sd['_model.gauss_params.features_dc'].float().numpy() + 0.5, 0, 1)
if len(means) > 400_000:
    keep = np.random.RandomState(0).choice(len(means), 400_000, replace=False)
    means, rgb = means[keep], rgb[keep]
arr = np.zeros(len(means), dtype=[('x','<f4'),('y','<f4'),('z','<f4'),
                                  ('red','u1'),('green','u1'),('blue','u1')])
arr['x'], arr['y'], arr['z'] = means.T
arr['red'], arr['green'], arr['blue'] = (rgb * 255).astype(np.uint8).T
with open('/content/garden_points.ply', 'wb') as f:
    f.write(b'ply\nformat binary_little_endian 1.0\n')
    f.write(f'element vertex {len(arr)}\n'.encode())
    f.write(b'property float x\nproperty float y\nproperty float z\n'
            b'property uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n')
    f.write(arr.tobytes())
print('seed points:', len(arr))

!rm -rf /content/out_garden_edited2
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-train splatfacto \
  --data /content/data/edited_garden --output-dir /content/out_garden_edited2 \
  --max-num-iterations 30000 --viewer.quit-on-train-completion True --vis tensorboard \
  blender-data --ply-path /content/garden_points.ply

In [ ]:
# Render the refit on the same cameras -> before/after side-by-side; save
import math
from pathlib import Path
from PIL import Image
from nerfstudio.utils.eval_utils import eval_setup
from nerfstudio.cameras.cameras import Cameras, CameraType
from diffusers.utils import export_to_video

meta = json.load(open('/content/data/edited_garden/transforms.json'))
CFG = sorted(glob.glob('/content/out_garden_edited2/**/config.yml', recursive=True),
             key=os.path.getmtime)[-1]
_, pipe2, _, _ = eval_setup(Path(CFG), test_mode='inference')
W, H = 832, 480
fx = W / (2 * math.tan(meta['camera_angle_x'] / 2)); fy = fx * (480/840) / (832/1296)
os.makedirs('/content/after_frames2', exist_ok=True)
for i, f in enumerate(meta['frames'], start=1):
    c2w = torch.tensor(f['transform_matrix'], dtype=torch.float32)[:3]
    cam = Cameras(camera_to_worlds=c2w[None], fx=fx, fy=fy, cx=W/2, cy=H/2,
                  width=W, height=H, camera_type=CameraType.PERSPECTIVE).to(pipe2.device)
    with torch.no_grad():
        out = pipe2.model.get_outputs_for_camera(cam)['rgb'].cpu().numpy()
    Image.fromarray((out * 255).clip(0, 255).astype('uint8')).save(
        f'/content/after_frames2/{i:04d}.png')

if not os.path.isdir('/content/orbit_frames'):
    shutil.copytree(f'{P6}/orbit_frames', '/content/orbit_frames')
before = [np.asarray(Image.open(p).convert('RGB'))
          for p in sorted(glob.glob('/content/orbit_frames/*.jpg'))]
after = [np.asarray(Image.open(p).convert('RGB'))
         for p in sorted(glob.glob('/content/after_frames2/*.png'))]
side = [Image.fromarray(np.hstack([b, a])) for b, a in zip(before, after)]
export_to_video(side, '/content/garden_before_after.mp4', fps=16)
shutil.copytree('/content/after_frames2', f'{P6}/after_frames', dirs_exist_ok=True)
shutil.copy('/content/garden_before_after.mp4', P6)
for d in glob.glob('/content/out_garden_edited2/*'):
    shutil.copytree(d, f'{P6}/out_garden_edited2/{os.path.basename(d)}', dirs_exist_ok=True)
side[40]   # right half must be sharp AND vase-free